# 07 — SABR, the swaption cube, and where Hagan's expansion breaks

## The model

For each expiry × tenor node the forward swap rate follows shifted SABR under its annuity measure:

$$d\tilde F_t = \alpha_t \tilde F_t^{\beta}\, dW_t, \qquad d\alpha_t = \nu\, \alpha_t\, dZ_t, \qquad dW\,dZ = \rho\,dt, \qquad \tilde F = F + s.$$

$\beta$ sets the backbone (how the ATM vol moves with the forward), $\rho$ the skew, $\nu$ the curvature (vol of
vol). $\beta$ and $s$ are desk policy; $\alpha, \rho, \nu$ are fitted per node.

Hagan, Kumar, Lesniewski and Woodward (2002) give the implied normal vol to second order (their eq. B.69):

$$\sigma_N(K) = \alpha\,\frac{(1-\beta)(f-k)}{f^{1-\beta}-k^{1-\beta}}\cdot\frac{\zeta}{\hat x(\zeta)}\cdot\left\{1 + \left[-\frac{\beta(2-\beta)\alpha^2}{24 f_{av}^{2-2\beta}} + \frac{\rho\alpha\nu\beta}{4 f_{av}^{1-\beta}} + \frac{2-3\rho^2}{24}\nu^2\right]T\right\}$$

with $\zeta = \frac{\nu}{\alpha}\frac{f^{1-\beta}-k^{1-\beta}}{1-\beta}$,
$\hat x(\zeta) = \ln\frac{\sqrt{1-2\rho\zeta+\zeta^2}+\zeta-\rho}{1-\rho}$ and $f_{av} = \sqrt{fk}$.

## A hand example (beta = 0)

With $\beta = 0$ the first factor is $\alpha$, $\zeta = (\nu/\alpha)(F - K)$, and $\zeta/\hat x = 1 - \rho\zeta/2 + O(\zeta^2)$. So

$$\sigma_{ATM} = \alpha\left(1 + \frac{(2-3\rho^2)\nu^2 T}{24}\right), \qquad \left.\frac{\partial\sigma_N}{\partial K}\right|_{F} \approx \frac{\rho\nu}{2}\left(1 + \frac{(2-3\rho^2)\nu^2 T}{24}\right).$$

For $\alpha = 85$ bp, $\rho = -0.3$, $\nu = 0.4$ and $T = 5$ you get an ATM vol of $85 \times 1.0577 = 89.90$ bp and a
skew of $-0.0635$ bp of vol per bp of strike. The C++ below must agree.

In [ ]:
import _util
from _util import run_demo
import numpy as np, pandas as pd, matplotlib.pyplot as plt
import gen_vol_quotes as g     # tools/ is on the path via _util: the generating surfaces of the sample screen

alpha, rho, nu, T, F = 0.0085, -0.3, 0.4, 5.0, 0.036
lift = 1 + (2 - 3 * rho**2) * nu**2 * T / 24
print(f"by hand:  ATM {alpha * lift * 1e4:.4f} bp, skew {rho * nu / 2 * lift:+.5f}")
sm = run_demo("rl_demo_vol", "smile", alpha, 0.0, rho, nu, 0.0, F, T, F - 0.0005, F + 0.0005, 3)
print(f"C++:      ATM {sm['normal_vol_bp'].iloc[1]:.4f} bp, skew {(sm['normal_vol_bp'].iloc[2] - sm['normal_vol_bp'].iloc[0]) / 10:+.5f}")

## What each parameter does

ATM vol held at 88 bp on a 1Y×10Y node (α re-solved each time). $\nu$ bends the wings up; $\rho$ tilts the smile;
$\beta$ moves the backbone, and in normal-vol terms a larger $\beta$ lifts the high-strike wing.

In [ ]:
F, T, ATM = 0.036, 1.0, 0.0088
def smile(beta=0.5, rho=-0.13, nu=0.41):
    a = g.alpha_from_atm(ATM, F, T, beta, rho, nu, 0.0)
    return run_demo("rl_demo_vol", "smile", a, beta, rho, nu, 0.0, F, T, 0.011, 0.061, 101)

fig, axes = plt.subplots(1, 3, figsize=(15, 4), sharey=True)
for v in (0.1, 0.4, 0.8):
    s = smile(nu=v); axes[0].plot(s["strike"] * 100, s["normal_vol_bp"], label=f"nu {v}")
for r in (-0.5, 0.0, 0.5):
    s = smile(rho=r); axes[1].plot(s["strike"] * 100, s["normal_vol_bp"], label=f"rho {r}")
for b in (0.0, 0.5, 1.0):
    s = smile(beta=b); axes[2].plot(s["strike"] * 100, s["normal_vol_bp"], label=f"beta {b}")
for ax, title in zip(axes, ["vol of vol", "correlation", "backbone"]):
    ax.set_title(title); ax.set_xlabel("strike, %"); ax.legend()
axes[0].set_ylabel("normal vol, bp"); plt.show()

## The screen, validated before any fit

`ratelib.vol.vol_quotes` runs the desk checks: a complete grid, units, offsets, ATM term-structure jumps, and
concave smiles SABR cannot fit. This cell needs no C++.

In [ ]:
import datetime as dt
from ratelib.vol import load_vol_quotes, validate_vol_quotes
from ratelib.vol.vol_quotes import default_path, summary

q = load_vol_quotes(default_path(), validate=False)
print(summary(q))
for w in validate_vol_quotes(q, today=dt.date(2026, 9, 4)):
    print("warning:", w)
q.atm_matrix()

## The fit

`vol::fitSabrSmile`, `vol::sabrInitialGuess` and `SabrNodeProblem::scaling` are in `cpp/vol/src/sabr_fit.cpp`.
The verdict must
read PASS with a max residual under 0.1 bp. Quotes are rounded to 0.01 bp, so expect a few thousandths. No node may
sit on a bound.

In [ ]:
from ratelib.vol import USD_POLICY, fit_nodes, load_fit_csv

rep = load_fit_csv(fit_nodes(USD_POLICY).to_csv(index=False))
print(rep.verdict())
rep.worst(5)[["id", "alpha", "rho", "nu", "rms_bp", "max_bp", "iterations", "status", "flags"]]

In [ ]:
piv = rep.pivot("max_bp")
fig, ax = plt.subplots(figsize=(8, 5))
im = ax.imshow(piv.values, cmap="viridis")
ax.set_xticks(range(len(piv.columns))); ax.set_xticklabels(piv.columns)
ax.set_yticks(range(len(piv.index))); ax.set_yticklabels(piv.index)
ax.set_xlabel("tenor"); ax.set_ylabel("expiry"); ax.set_title("max |residual| per node, bp")
fig.colorbar(im); plt.show()

## Where it breaks: the implied density

The risk-neutral density is $p(K) = \partial^2 C/\partial K^2$ (Breeden–Litzenberger). A negative density means a
butterfly with negative value: static arbitrage. Hagan's formula is an expansion in $\alpha^2 T f^{2\beta-2}$ and
$\nu^2 T$, and it fails in two recognisable places:

* at **low strikes when $\beta > 0$**: the $-\beta(2-\beta)\alpha^2/(24 f_{av}^{2-2\beta})$ term diverges as $K \to 0$;
* for **long expiries with large $\nu$**: $\nu^2 T$ is no longer small.

The two cells below use the *generating* parameters of the sample screen and work now. 10Y×20Y is benign. 10Y×30Y
has $\nu = 0.70$. It fits to the same precision as every other node, yet its density goes negative about 170 bp
below the forward, inside one standard deviation. A perfect fit and an arbitrage-free smile are different things.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
for (e, t), colour in [(("10Y", "20Y"), "C0"), (("10Y", "30Y"), "C3")]:
    p = g.node_parameters(e, t)
    d = run_demo("rl_demo_vol", "density", p["alpha"], p["beta"], p["rho"], p["nu"], p["shift"], p["forward"], p["expiry_years"], 0.002, p["forward"] + 0.06, 400)
    ax.plot(d["strike"] * 100, d["density"], colour, label=f"{e}x{t}: nu = {p['nu']:.2f}")
    chk = run_demo("rl_demo_vol", "check", p["alpha"], p["beta"], p["rho"], p["nu"], p["shift"], p["forward"], p["expiry_years"]).iloc[0]
    if pd.notna(chk["first_negative_below"]):
        ax.axvline(chk["first_negative_below"] * 100, color=colour, ls=":", label=f"{e}x{t}: first negative density {chk['first_negative_below'] * 100:.2f}%")
    ax.axvline(p["forward"] * 100, color=colour, lw=0.5)
ax.axhline(0.0, color="k", lw=0.6); ax.set_ylim(-15, 25)
ax.set_xlabel("strike, %"); ax.set_ylabel("density per unit rate"); ax.legend(fontsize=8); plt.show()

## The whole cube: where does the usable smile end?

For every node, this is the first strike below the forward where the density turns negative, scanning over
$F \pm 4\sigma\sqrt T$ (floored at 1 bp because $\beta > 0$). The number rises with expiry: every 10Y node goes negative
somewhere above zero, and only the stressed node does so close to the money. Once your fit runs,
`ratelib.vol.arbitrage_scan()` gives the same map from the *fitted* parameters, through `CubeReport.negative_density_map()`.

In [ ]:
rows = []
for e in g.EXPIRIES:
    for t in g.TENORS:
        p = g.node_parameters(e, t)
        c = run_demo("rl_demo_vol", "check", p["alpha"], p["beta"], p["rho"], p["nu"], p["shift"], p["forward"], p["expiry_years"], 4.0, 801).iloc[0]
        rows.append((e, t, c["first_negative_below"] * 100 if pd.notna(c["first_negative_below"]) else np.nan))
kstar = pd.DataFrame(rows, columns=["expiry", "tenor", "k_star_pct"]).pivot(index="expiry", columns="tenor", values="k_star_pct").reindex(index=g.EXPIRIES, columns=g.TENORS)
kstar.round(2)

In [ ]:
from ratelib.vol import arbitrage_scan
arb = arbitrage_scan(USD_POLICY)            # fitted parameters
rep_arb = load_fit_csv(fit_nodes(USD_POLICY).to_csv(index=False), arbitrage=arb)
print(rep_arb.verdict())
rep_arb.negative_density_map().round(2)

## What v1 does about it, and what production does

v1 **reports** and does not repair. The calibration report lists, per node, the first strike where the smile stops
being usable. Anything that integrates over the wing (CMS replication in v2, a callable's exercise boundary deep
out of the money) must not go below that strike without a fix.

The production fix is *arbitrage-free SABR* (Hagan, Kumar, Lesniewski and Woodward, 2014, Wilmott). It solves the
effective one-dimensional forward equation for the density, which is positive by construction and reprices the
same near-the-money smile. The cheaper alternative is to replace the wing below the first bad strike with an
arbitrage-free extrapolation. Either changes every low-strike price, which is why the diagnostic comes first.

Questions to check: which term in (B.69) diverges as $K \to 0$ and why only for $\beta > 0$; why the density is
$\partial^2 C/\partial K^2$ of the *undiscounted* price; why fitting $(\rho, \nu)$ with $\alpha$ tied to ATM gives a
better-conditioned problem than fitting all three.